# Solución Big Data con Apache Spark y Arquitectura Medallion
## Retrasos logísticos en DataCo Global
**Curso:** Base de Datos Avanzadas y Big Data · **Grupo 02** · UPN · Período 2026-2
**Plataforma:** Databricks (Apache Spark, tablas Delta, gráficos y dashboard de Databricks)

### Cobertura de los requisitos del curso
| Requisito del profesor | Dónde se cumple |
|---|---|
| Identificación de los datos | Parte 1 |
| Arquitectura (gráfico) | Parte 1 · archivo `Arquitectura_Medallion_DataCo.png` |
| Procesamiento con Databricks: dataset en DataFrames | Parte 3 (Bronze) |
| Análisis exploratorio (EDA) | Se entrega por separado |
| Limpieza y preparación con operaciones sobre DataFrames | Partes 4 y 5 |
| Arquitectura Medallion | Partes 3 (Bronze), 4 (Silver) y 6 (Gold) |
| Persistencia en tablas de Databricks | Partes 3, 4 y 6 |
| Indicadores, visualizaciones y dashboard | Partes 6 y 7 (tablas para graficar con el **+**) |
| *Extra:* reconciliación con la solución BI | Parte 8 |
| *Extra:* experimento de escalabilidad EXP-02 | Parte 9 |
| *Extra:* modelo predictivo opcional | Parte 10 |

## Parte 1. Identificación de los datos y arquitectura
**Fuente:** `DataCoSupplyChainDataset.csv`, el **mismo dataset de la solución BI** (SQL Server + Power BI), reutilizado para comparar ambas soluciones con los mismos datos.

| Característica | Valor |
|---|---|
| Filas | 180 519 (cada fila es un ítem de pedido) |
| Columnas | 53 |
| Pedidos únicos / clientes únicos | 65 752 / 20 652 |
| Periodo de pedidos | 01-01-2015 a 31-01-2018 |
| Codificación | ISO-8859-1 |
| Variable objetivo del negocio | `late_delivery_risk` (1 = entrega tardía) |

**Problemática:** las entregas tardías afectan la satisfacción y los costos logísticos. El análisis responde las mismas preguntas que el BI:
- **Q1.** ¿Cuántas entregas son tardías y cuál es el promedio de días reales vs. programados?
- **Q2.** ¿Qué modo de envío concentra más retrasos?
- **Q3.** ¿En qué región de destino y categoría de producto se concentran los retrasos?
- **Q4.** ¿Cómo evolucionan las entregas tardías y la ganancia por año y mes?

**Arquitectura:** CSV → Volume de Unity Catalog → **Bronze** (crudo) → **Silver** (limpio) → **Gold** (indicadores) → Dashboard. Ver el gráfico `Arquitectura_Medallion_DataCo.png`; para verlo en el notebook, súbalo al Volume `datos` y ejecute la celda siguiente.

In [ ]:
# Opcional: mostrar el diagrama de arquitectura (subir el PNG al Volume 'datos')
try:
    from IPython.display import Image, display as ipy_display
    ipy_display(Image(filename="/Volumes/workspace/default/datos/Arquitectura_Medallion_DataCo.png", width=900))
except Exception as e:
    print("Suba Arquitectura_Medallion_DataCo.png al Volume 'datos' para verlo aquí:", e)

## Parte 2. Preparar el entorno
1. **Catalog** → `workspace` → `default` → **Create** → **Volume** → nombre `datos`.
2. Dentro del volume: **Upload to this volume** → `DataCoSupplyChainDataset.csv`.

Si usó otra ruta, cambie `RUTA_CSV`.

In [ ]:
import re, time, datetime, statistics
import pandas as pd
from pyspark.sql import functions as F

RUTA_CSV = "/Volumes/workspace/default/datos/DataCoSupplyChainDataset.csv"
BRONZE, SILVER = "bronze_supply_chain", "silver_supply_chain"
FMT_FECHA = "M/d/yyyy H:mm"

display(dbutils.fs.ls("/Volumes/workspace/default/datos/"))

## Parte 3. Capa Bronze: ingesta del dataset en DataFrames
Bronze guarda los datos **tal como llegan**, con dos únicos cambios técnicos:
- Nombres de columna a `snake_case` (Delta no admite espacios ni paréntesis en los nombres).
- Columnas de **linaje**: `source_file` (archivo de origen) e `ingest_ts` (momento de carga).

El CSV está en **ISO-8859-1**; sin la opción `encoding` los acentos se corrompen.

In [ ]:
t0 = time.perf_counter()
df_csv = (spark.read
  .format("csv")
  .option("header", "true")
  .option("inferSchema", "true")
  .option("encoding", "ISO-8859-1")
  .load(RUTA_CSV))

cols_originales = df_csv.columns
print("Columnas originales:", len(cols_originales))
print(cols_originales[:6], "...")

In [ ]:
def limpiar_nombre(n):
    # 'Days for shipping (real)' -> 'days_for_shipping_real'
    return re.sub(r"[^0-9a-zA-Z]+", "_", n.strip()).strip("_").lower()

df_bronze = (df_csv
  .select(*[F.col(f"`{c}`") for c in cols_originales],
          F.col("_metadata.file_path").alias("source_file"))
  .toDF(*[limpiar_nombre(c) for c in cols_originales], "source_file")
  .withColumn("ingest_ts", F.current_timestamp()))

df_bronze.write.mode("overwrite").saveAsTable(BRONZE)

b = spark.table(BRONZE)
FILAS_BRONZE = b.count()
print(f"Bronze guardada: {FILAS_BRONZE:,} filas | {len(b.columns)} columnas | {time.perf_counter()-t0:.1f} s")

In [ ]:
b.printSchema()
display(b.limit(5))

## Parte 4. Capa Silver: limpieza y preparación
Cada paso es una operación sobre DataFrames y se registra cuántas filas quedan, para demostrar el proceso y no solo la carga.

Los problemas que corrige esta capa vienen del EDA (entregado por separado):
- Espacios sobrantes en `category_name`, `department_name`, `order_region` y `product_name`.
- `customer_state` con códigos postales en lugar del estado.
- `customer_country` con la variante `EE. UU.`.
- Días reales que no coinciden con la diferencia entre fechas.
- Columnas personales, casi vacías o constantes.

| Paso | Operación Spark | Motivo |
|---|---|---|
| 1 | `drop` | Eliminar columnas personales, nulas o constantes |
| 2 | `trim` + vacío → `null` (`select`, `when`) | Espacios sobrantes |
| 3 | `na.drop(subset=...)` | Quitar filas sin campos críticos |
| 4 | `dropDuplicates` | Unicidad de la clave |
| 5 | `when/otherwise`, `rlike`, `na.fill` | Estandarizar país y estado |
| 6 | `to_timestamp`, `cast` | Tipos correctos |
| 7 | `filter` | Reglas de validez; lo rechazado se cuenta |
| 8 | `withColumn` | Banderas de calidad y columnas derivadas |
| 9 | `withColumnRenamed` | Nombre más claro |

In [ ]:
log = []
def reg(nombre, d):
    log.append({"paso": nombre, "filas": d.count(), "columnas": len(d.columns)})
    return d

s0 = reg("0. Bronze (entrada)", spark.table(BRONZE))

# 1. drop
A_ELIMINAR = ["customer_email", "customer_password", "customer_street", "customer_fname", "customer_lname",
              "customer_zipcode", "order_zipcode", "product_description", "product_image", "product_status"]
s1 = reg("1. drop de columnas", s0.drop(*A_ELIMINAR))

# 2. trim y vacíos a null en todas las columnas de texto
cols_txt = [c for c, t in s1.dtypes if t == "string" and c != "source_file"]
s2 = reg("2. trim / vacío → null", s1.select(*[
    F.when(F.trim(F.col(c)) == "", None).otherwise(F.trim(F.col(c))).alias(c) if c in cols_txt else F.col(c)
    for c in s1.columns]))

# 3. na.drop en campos críticos
CRITICOS = ["order_id", "order_item_id", "customer_id", "order_date_dateorders", "shipping_date_dateorders",
            "shipping_mode", "late_delivery_risk", "days_for_shipping_real", "days_for_shipment_scheduled", "sales"]
s3 = reg("3. na.drop (campos críticos)", s2.na.drop(subset=CRITICOS))

# 4. duplicados por clave
s4 = reg("4. dropDuplicates (order_id, order_item_id)", s3.dropDuplicates(["order_id", "order_item_id"]))

In [ ]:
# 5. estandarización: país y estado
s5 = (s4
  .withColumn("customer_country",
              F.when(F.col("customer_country") == "EE. UU.", F.lit("USA")).otherwise(F.col("customer_country")))
  .withColumn("customer_state",
              F.when(F.col("customer_state").rlike("^[A-Z]{2}$"), F.col("customer_state")))   # inválido -> null
  .na.fill({"customer_state": "DESCONOCIDO"}))
s5 = reg("5. estandarizar país/estado + na.fill", s5)

# 6. fechas y tipos
s6 = (s5
  .withColumn("order_date",    F.to_timestamp("order_date_dateorders", FMT_FECHA))
  .withColumn("shipping_date", F.to_timestamp("shipping_date_dateorders", FMT_FECHA))
  .drop("order_date_dateorders", "shipping_date_dateorders")
  .withColumn("late_delivery_risk", F.col("late_delivery_risk").cast("int"))
  .withColumn("order_item_discount_rate", F.col("order_item_discount_rate").cast("double")))
s6 = reg("6. to_timestamp + cast", s6)

# 7. reglas de validez
reglas = ((F.col("sales") > 0) & (F.col("order_item_quantity") > 0) & (F.col("product_price") > 0) &
          (F.col("days_for_shipping_real") >= 0) & (F.col("days_for_shipment_scheduled") >= 0) &
          (F.col("shipping_date") >= F.col("order_date")) &
          (F.col("order_item_discount_rate").between(0, 1)) & (F.col("late_delivery_risk").isin(0, 1)))
rechazados = s6.filter(~reglas)
print("Filas rechazadas por reglas de validez:", rechazados.count())
s7 = reg("7. filter (reglas de validez)", s6.filter(reglas))

In [ ]:
# 8. banderas de calidad y columnas derivadas
q1_v, q3_v = s7.approxQuantile("sales", [0.25, 0.75], 0.001)
lim_sup_ventas = q3_v + 1.5 * (q3_v - q1_v)

dias_por_fechas = F.floor((F.unix_timestamp("shipping_date") - F.unix_timestamp("order_date")) / 86400)

s8 = (s7
  .withColumn("flag_dias_vs_fechas",       (dias_por_fechas != F.col("days_for_shipping_real")).cast("int"))
  .withColumn("flag_total_inconsistente",
              (F.abs(F.col("order_item_total") - (F.col("sales") - F.col("order_item_discount"))) > 0.01).cast("int"))
  .withColumn("flag_perdida",              (F.col("benefit_per_order") < 0).cast("int"))
  .withColumn("flag_outlier_ventas",       (F.col("sales") > lim_sup_ventas).cast("int"))
  .withColumn("order_year",  F.year("order_date"))
  .withColumn("order_month", F.month("order_date"))
  .withColumn("year_month",  F.date_format("order_date", "yyyy-MM"))
  .withColumn("shipping_delay_days", F.col("days_for_shipping_real") - F.col("days_for_shipment_scheduled"))
  .withColumn("entrega_tardia_regla",
              (F.col("days_for_shipping_real") > F.col("days_for_shipment_scheduled")).cast("int")))
s8 = reg("8. banderas de calidad + derivadas", s8)

# 9. renombrar
s9 = reg("9. withColumnRenamed (type → payment_type)", s8.withColumnRenamed("type", "payment_type"))

s9.write.mode("overwrite").saveAsTable(SILVER)
print("Silver guardada:", SILVER)

In [ ]:
# Registro del proceso de limpieza (va al informe)
display(pd.DataFrame(log))

### 4.1 Reporte de calidad: Bronze vs. Silver

In [ ]:
COLS_TXT = ["category_name", "department_name", "order_region", "product_name"]
cond_espacios = " OR ".join(f"{c} != trim({c})" for c in COLS_TXT)

def metricas(d):
    return {
      "filas": d.count(),
      "columnas": len(d.columns),
      "claves_duplicadas": d.groupBy("order_id", "order_item_id").count().filter("count > 1").count(),
      "filas_con_espacios_sobrantes": d.filter(cond_espacios).count(),
      "customer_state_con_codigo_postal": d.filter(F.col("customer_state").rlike("^[0-9]+$")).count(),
      "customer_country_variante_EE_UU": d.filter(F.col("customer_country") == "EE. UU.").count(),
    }

silver = spark.table(SILVER)
mb, ms = metricas(spark.table(BRONZE)), metricas(silver)
reporte = pd.DataFrame({"metrica": list(mb.keys()), "bronze": list(mb.values()), "silver": list(ms.values())})
display(reporte)

print(f"Completitud (Silver / Bronze): {ms['filas'] / mb['filas'] * 100:.2f} %  (umbral CA-01 >= 99.9 %)")

Las filas con **banderas** (`flag_*`) no se eliminan: se conservan para que el análisis decida si las excluye.

In [ ]:
display(silver.agg(*[F.sum(c).alias(c) for c in
        ["flag_dias_vs_fechas", "flag_total_inconsistente", "flag_perdida", "flag_outlier_ventas"]]))

## Parte 5. Operaciones con DataFrames (guía de laboratorio, Partes 1–4 y 7)
Se aplican, sobre la capa Silver, las operaciones de la guía del profesor.

| Operación | Parte de la guía | Subparte |
|---|---|---|
| pandas vs. Spark, `createDataFrame` | 1 y 2 | 5.1 |
| `filter` / `where`, `select`, `selectExpr`, `drop`, `withColumnRenamed` | 4 y 6 | 5.2 |
| `join` | 3 | 5.3 |
| `union`, `distinct` | 3 | 5.4 |
| `describe`, `sample` | 6 | 5.5 |
| `struct` y notación de punto | 7 | 5.6 |

In [ ]:
df = spark.table(SILVER)          # DataFrame base para el resto del notebook

### 5.1 pandas vs. Spark y `createDataFrame`

In [ ]:
muestra_pd = df.select("order_id", "shipping_mode", "late_delivery_risk").limit(5).toPandas()   # vive en el driver
display(muestra_pd)

df_desde_pandas = spark.createDataFrame(muestra_pd)

# Tabla de apoyo con esquema en texto (clasificación ilustrativa definida por el grupo, no del dataset)
modos = spark.createDataFrame(
    [("Same Day", "Express"), ("First Class", "Express"), ("Second Class", "Rápido"), ("Standard Class", "Estándar")],
    schema="shipping_mode STRING, categoria_servicio STRING")
display(modos)

**Pregunta 1:** ¿dónde se almacena y procesa `muestra_pd`? ¿Qué ventaja tiene Spark al crecer el volumen (ver Parte 9)?

### 5.2 Filtrar y proyectar

In [ ]:
print("First Class tardías:", df.filter("late_delivery_risk = 1 AND shipping_mode = 'First Class'").count())

latam_altas = df.where((F.col("market") == "LATAM") & (F.col("sales") > 200))
display(latam_altas.select("order_id", "market", "category_name", "sales").limit(10))

display(df.selectExpr("order_id", "upper(shipping_mode) AS modo", "round(sales, 1) AS ventas_redondeadas",
                      "days_for_shipping_real - days_for_shipment_scheduled AS brecha_dias").limit(10))

display(df.select("order_id", "sales", "benefit_per_order")
          .withColumnRenamed("benefit_per_order", "ganancia").drop("sales").limit(5))

**Pregunta 2:** `filter` y `select` son transformaciones; `count()` y `display()` son acciones. ¿Qué línea de la celda anterior dispara la ejecución?

### 5.3 `join`

In [ ]:
por_modo = (df.groupBy("shipping_mode")
              .agg(F.count("*").alias("entregas"), F.round(F.avg("late_delivery_risk") * 100, 2).alias("pct_tardias")))
display(por_modo.join(modos, on="shipping_mode", how="inner").orderBy(F.desc("pct_tardias")))

display(df.join(modos, on="shipping_mode", how="inner").groupBy("categoria_servicio")
          .agg(F.count("*").alias("entregas"), F.round(F.avg("late_delivery_risk") * 100, 2).alias("pct_tardias")))

### 5.4 `union` y `distinct`

In [ ]:
d2017, d2018 = df.filter("order_year = 2017"), df.filter("order_year = 2018")
union_df = d2017.union(d2018)
print("2017:", d2017.count(), "| 2018:", d2018.count(), "| union:", union_df.count())

claves = union_df.select("order_id", "order_item_id")
print("claves + claves:         ", claves.union(claves).count())
print("claves + claves distinct:", claves.union(claves).distinct().count())

### 5.5 `describe` y `sample`

In [ ]:
display(df.describe("sales", "benefit_per_order", "days_for_shipping_real", "days_for_shipment_scheduled"))
muestra10 = df.sample(0.10, seed=42)
print("Muestra ~10 %:", muestra10.count(), "de", df.count())

### 5.6 Columna anidada (`struct`) y notación de punto

In [ ]:
df_struct = df.select("order_id", F.struct("order_country", "order_state", "order_city").alias("destino"))
df_struct.printSchema()
display(df_struct.select("order_id", "destino.order_country", "destino.order_city").limit(5))
display(df_struct.select("order_id", "destino.*").limit(5))

**Pregunta 3:** ¿qué diferencia hay entre `select("destino")` y `select("destino.*")`?

## Parte 6. Capa Gold: indicadores de negocio
Gold contiene tablas **pequeñas y listas para consumir**: una por cada necesidad de información. Cada tabla se guarda como Delta y se muestra en la Parte 7 para graficarla con el **+**.

“Ganancia” = `benefit_per_order` (idéntica a `order_profit_per_order` en este dataset).

In [ ]:
df = spark.table(SILVER)          # DataFrame base de Silver
TOTAL = df.count()

# Indicadores Q1–Q4 (se reutilizan en la reconciliación y en el EXP-02)
def q1(t):
    return t.agg(F.count("*").alias("total_entregas"),
                 F.sum("late_delivery_risk").alias("entregas_tardias"),
                 F.round(F.avg("late_delivery_risk") * 100, 2).alias("pct_tardias"),
                 F.round(F.avg("days_for_shipping_real"), 2).alias("dias_reales_prom"),
                 F.round(F.avg("days_for_shipment_scheduled"), 2).alias("dias_prog_prom"))

def q2(t):
    return (t.groupBy("shipping_mode")
             .agg(F.count("*").alias("entregas"),
                  F.round(F.avg("days_for_shipment_scheduled"), 2).alias("dias_prog_prom"),
                  F.round(F.avg("days_for_shipping_real"), 2).alias("dias_reales_prom"),
                  F.round(F.avg("late_delivery_risk") * 100, 2).alias("pct_tardias"))
             .orderBy(F.desc("pct_tardias")))

def q3(t):
    return (t.groupBy("order_region", "category_name")
             .agg(F.sum("late_delivery_risk").alias("entregas_tardias"))
             .orderBy(F.desc("entregas_tardias")))

def q4(t):
    return (t.groupBy("order_year", "order_month")
             .agg(F.sum("late_delivery_risk").alias("entregas_tardias"),
                  F.round(F.sum("benefit_per_order"), 2).alias("ganancia_total"))
             .orderBy("order_year", "order_month"))

QUERIES = {"Q1": q1, "Q2": q2, "Q3": q3, "Q4": q4}

# Apoyo para las tablas Gold
def resumen_retraso(t, *cols):
    return (t.groupBy(*cols)
             .agg(F.count("*").alias("entregas"),
                  F.sum("late_delivery_risk").alias("entregas_tardias"),
                  F.round(F.avg("late_delivery_risk") * 100, 2).alias("pct_tardias"))
             .orderBy(F.desc("pct_tardias")))

def ventas_ganancia():
    return [F.round(F.sum("sales"), 2).alias("ventas"), F.round(F.sum("benefit_per_order"), 2).alias("ganancia")]

In [ ]:
GOLD = {
  # A. Panorama
  "gold_kpi_global": q1(df),
  "gold_estado_entrega": (df.groupBy("delivery_status").agg(F.count("*").alias("entregas"))
                            .withColumn("pct", F.round(F.col("entregas") / F.lit(TOTAL) * 100, 2))
                            .orderBy(F.desc("entregas"))),
  # B. Por dónde se retrasa (Q2, Q3)
  "gold_kpi_modo_envio": (df.groupBy("shipping_mode")
                            .agg(F.count("*").alias("entregas"),
                                 F.sum("late_delivery_risk").alias("entregas_tardias"),
                                 F.round(F.avg("late_delivery_risk") * 100, 2).alias("pct_tardias"),
                                 F.round(F.avg("days_for_shipment_scheduled"), 2).alias("dias_programados"),
                                 F.round(F.avg("days_for_shipping_real"), 2).alias("dias_reales"))
                            .withColumn("brecha_dias", F.round(F.col("dias_reales") - F.col("dias_programados"), 2))
                            .orderBy(F.desc("pct_tardias"))),
  "gold_mercado": (resumen_retraso(df, "market")
                     .join(df.groupBy("market").agg(*ventas_ganancia()), "market")
                     .orderBy(F.desc("entregas_tardias"))),
  "gold_mercado_modo": (resumen_retraso(df, "market", "shipping_mode")
                          .join(df.groupBy("market", "shipping_mode").agg(*ventas_ganancia()),
                                ["market", "shipping_mode"])
                          .orderBy("market", "shipping_mode")),
  "gold_region": resumen_retraso(df, "order_region").orderBy(F.desc("entregas_tardias")),
  "gold_pais": resumen_retraso(df, "order_country").filter(F.col("entregas") >= 500),
  "gold_categoria_tardias": resumen_retraso(df, "category_name").orderBy(F.desc("entregas_tardias")),
  "gold_region_categoria": (df.groupBy("order_region", "category_name")
                              .agg(F.sum("late_delivery_risk").alias("entregas_tardias"), F.count("*").alias("entregas"))
                              .withColumn("pct_tardias", F.round(F.col("entregas_tardias") / F.col("entregas") * 100, 2))
                              .withColumn("categoria_region", F.concat_ws(" · ", "order_region", "category_name"))
                              .orderBy(F.desc("entregas_tardias"))),
  "gold_departamento": resumen_retraso(df, "department_name"),
  # C. Cuándo se retrasa (Q4)
  "gold_serie_mensual": (df.groupBy("year_month")
                           .agg(F.count("*").alias("entregas"),
                                F.sum("late_delivery_risk").alias("entregas_tardias"),
                                F.round(F.avg("late_delivery_risk") * 100, 2).alias("pct_tardias"),
                                *ventas_ganancia())
                           .orderBy("year_month")),
  "gold_anual": (df.groupBy("order_year")
                   .agg(F.count("*").alias("entregas"),
                        F.sum("late_delivery_risk").alias("entregas_tardias"),
                        F.round(F.avg("late_delivery_risk") * 100, 2).alias("pct_tardias"),
                        *ventas_ganancia(),
                        F.countDistinct("order_month").alias("meses_con_datos"))
                   .orderBy("order_year")),
  "gold_estacionalidad": resumen_retraso(df, "order_month").orderBy("order_month"),
  # D. Cuánto cuesta
  "gold_impacto_entrega": (df.withColumn("tipo_entrega", F.when(F.col("late_delivery_risk") == 1, "Tardía").otherwise("No tardía"))
                             .groupBy("tipo_entrega")
                             .agg(F.count("*").alias("entregas"),
                                  F.round(F.sum("sales"), 2).alias("ventas"),
                                  F.round(F.sum("benefit_per_order"), 2).alias("ganancia_total"),
                                  F.round(F.avg("benefit_per_order"), 2).alias("ganancia_prom_item"),
                                  F.round(F.avg((F.col("benefit_per_order") < 0).cast("int")) * 100, 2).alias("pct_con_perdida"))),
  "gold_categoria_ganancia": (df.groupBy("category_name")
                                .agg(*ventas_ganancia(), F.sum("order_item_quantity").alias("unidades"))
                                .withColumn("margen_pct", F.round(F.col("ganancia") / F.col("ventas") * 100, 2))
                                .orderBy(F.desc("ganancia"))),
  "gold_mercado_anio": (df.groupBy("market", "order_year").agg(*ventas_ganancia()).orderBy("market", "order_year")),
  # E. Quiénes y qué
  "gold_segmento": (df.groupBy("customer_segment")
                      .agg(F.countDistinct("customer_id").alias("clientes"),
                           F.count("*").alias("entregas"),
                           F.round(F.avg("late_delivery_risk") * 100, 2).alias("pct_tardias"),
                           F.round(F.sum("sales"), 2).alias("ventas"),
                           F.round(F.avg("sales"), 2).alias("ticket_prom_item"))
                      .orderBy(F.desc("ventas"))),
  "gold_top_productos": (df.groupBy("product_name")
                           .agg(F.round(F.sum("sales"), 2).alias("ventas"),
                                F.sum("order_item_quantity").alias("unidades"),
                                F.round(F.avg("late_delivery_risk") * 100, 2).alias("pct_tardias"))
                           .orderBy(F.desc("ventas")).limit(10)),
  "gold_estado_pedido": (df.groupBy("order_status").agg(F.count("*").alias("pedidos_items"))
                           .withColumn("pct", F.round(F.col("pedidos_items") / F.lit(TOTAL) * 100, 2))
                           .orderBy(F.desc("pedidos_items"))),
  "gold_tipo_pago": resumen_retraso(df, "payment_type").orderBy(F.desc("entregas")),
  "gold_brecha_dias": (df.withColumn("brecha_dias", F.col("days_for_shipping_real") - F.col("days_for_shipment_scheduled"))
                         .groupBy("brecha_dias").agg(F.count("*").alias("entregas")).orderBy("brecha_dias")),
  # F. Calidad de datos (Bronze vs. Silver)
  "gold_calidad": spark.createDataFrame(reporte),
}

for nombre, d in GOLD.items():
    d.write.mode("overwrite").saveAsTable(nombre)
print(f"{len(GOLD)} tablas Gold guardadas:")
print(", ".join(GOLD.keys()))

## Parte 7. Indicadores y visualizaciones
Cada celda muestra **una tabla Gold** con `display()`, lista para graficar sin escribir código:

1. Ejecute la celda y espere la tabla.
2. Clic en el **+** junto a la pestaña *Table* → **Visualization**.
3. Configure el gráfico según la línea **Gráfico sugerido** (eje X, eje Y, series).
4. Con las tablas y gráficos terminados, arme el dashboard.

Las tablas responden las preguntas del negocio (**Q1–Q4**) y las amplían:

| Bloque | Tablas | Responde |
|---|---|---|
| A. Panorama | 7.1 y 7.2 | Q1 |
| B. Por dónde se retrasa | 7.3 a 7.10 | Q2 y Q3 |
| C. Cuándo se retrasa | 7.11 a 7.13 | Q4 |
| D. Cuánto cuesta | 7.14 a 7.16 | Impacto en ventas y ganancia |
| E. Quiénes y qué | 7.17 a 7.21 | Clientes, productos y pedidos |
| F. Calidad | 7.22 | Resultado de la limpieza |

### A. Panorama general

**7.1 Indicadores globales (Q1)** · *Necesidad:* ¿cuántas entregas hay, cuántas son tardías y cuánto difieren los días reales de los programados?
**Gráfico sugerido (+):** Contador (Counter), uno por cada valor: `total_entregas`, `entregas_tardias`, `pct_tardias`, `dias_reales_prom`, `dias_prog_prom`.

In [ ]:
display(spark.table("gold_kpi_global"))

**7.2 Estado de las entregas** · *Necesidad:* ¿qué proporción llega a tiempo, adelantada o tarde, o se cancela?
**Gráfico sugerido (+):** Circular (Pie) · Categoría = `delivery_status` · Valor = `entregas`.

In [ ]:
display(spark.table("gold_estado_entrega"))

### B. ¿Por dónde se retrasa? (Q2 y Q3)

**7.3 Retrasos por modo de envío (Q2)** · *Necesidad:* ¿qué modo de envío concentra más retrasos y cuánto se aleja del plazo prometido?
**Gráfico sugerido (+):** Barras · X = `shipping_mode` · Y = `pct_tardias`. Segundo gráfico: barras agrupadas con `dias_programados` y `dias_reales`.

In [ ]:
display(spark.table("gold_kpi_modo_envio"))

**7.4 Retrasos por mercado** · *Necesidad:* ¿qué mercado tiene más entregas tardías, en cantidad y en proporción?
**Gráfico sugerido (+):** Barras · X = `market` · Y = `entregas_tardias` (y otro con `pct_tardias`).

In [ ]:
display(spark.table("gold_mercado"))

**7.5 Mercado × modo de envío** · *Necesidad:* ¿el problema del modo de envío es igual en todos los mercados?
**Gráfico sugerido (+):** Barras agrupadas · X = `market` · Y = `pct_tardias` · Agrupar por = `shipping_mode` (o Mapa de calor: X = `shipping_mode`, Y = `market`, Color = `pct_tardias`).

In [ ]:
display(spark.table("gold_mercado_modo"))

**7.6 Retrasos por región de destino (Q3)** · *Necesidad:* ¿qué regiones concentran más retrasos?
**Gráfico sugerido (+):** Barras horizontales · X = `entregas_tardias` · Y = `order_region`.

In [ ]:
display(spark.table("gold_region"))

**7.7 Top 15 países por % de retrasos** · *Necesidad:* ¿qué países tienen la peor puntualidad? (se exigen al menos 500 entregas por país)
**Gráfico sugerido (+):** Barras · X = `order_country` · Y = `pct_tardias`.

In [ ]:
display(spark.table("gold_pais").limit(15))

**7.8 Top 10 categorías con más entregas tardías (Q3)** · *Necesidad:* ¿qué categorías de producto concentran los retrasos?
**Gráfico sugerido (+):** Barras horizontales · X = `entregas_tardias` · Y = `category_name`.

In [ ]:
display(spark.table("gold_categoria_tardias").limit(10))

**7.9 Región × categoría (Q3 completa)** · *Necesidad:* ¿qué combinaciones de región y categoría tienen más entregas tardías?
**Gráfico sugerido (+):** Barras · X = `categoria_region` · Y = `entregas_tardias`.

In [ ]:
display(spark.table("gold_region_categoria").limit(15))

**7.10 Retrasos por departamento de producto** · *Necesidad:* ¿algún departamento es más problemático?
**Gráfico sugerido (+):** Barras · X = `department_name` · Y = `pct_tardias`.

In [ ]:
display(spark.table("gold_departamento"))

### C. ¿Cuándo se retrasa? (Q4)

**7.11 Evolución mensual (Q4)** · *Necesidad:* ¿cómo evolucionan las entregas tardías, las ventas y la ganancia mes a mes?
**Gráfico sugerido (+):** Líneas · X = `year_month` · Y = `entregas_tardias` y `ganancia` (o `pct_tardias`).

In [ ]:
display(spark.table("gold_serie_mensual"))

**7.12 Resumen anual** · *Necesidad:* ¿mejora o empeora la puntualidad de un año a otro? Los pedidos terminan en enero de 2018, así que **2018 solo tiene un mes** y no es comparable con los años completos (ver `meses_con_datos`).
**Gráfico sugerido (+):** Barras · X = `order_year` · Y = `pct_tardias`.

In [ ]:
display(spark.table("gold_anual"))

**7.13 Estacionalidad por mes del año** · *Necesidad:* ¿hay meses del año con más retrasos?
**Gráfico sugerido (+):** Barras · X = `order_month` · Y = `pct_tardias`.

In [ ]:
display(spark.table("gold_estacionalidad"))

### D. ¿Cuánto cuesta? (ventas y ganancia)

**7.14 Entregas tardías vs. no tardías** · *Necesidad:* ¿las entregas tardías venden y ganan distinto? ¿tienen más pérdidas?
**Gráfico sugerido (+):** Barras agrupadas · X = `tipo_entrega` · Y = `ganancia_prom_item` (y otro con `pct_con_perdida`).

In [ ]:
display(spark.table("gold_impacto_entrega"))

**7.15 Ventas y ganancia por categoría** · *Necesidad:* ¿qué categorías generan más ganancia y cuáles pierden dinero?
**Gráfico sugerido (+):** Barras horizontales · X = `ganancia` · Y = `category_name`.

In [ ]:
display(spark.table("gold_categoria_ganancia"))

**7.16 Ventas y ganancia por mercado y año** · *Necesidad:* ¿cómo evoluciona el negocio en cada mercado?
**Gráfico sugerido (+):** Barras agrupadas · X = `order_year` · Y = `ventas` · Agrupar por = `market`.

In [ ]:
display(spark.table("gold_mercado_anio"))

### E. ¿Quiénes y qué? (clientes, productos y pedidos)

**7.17 Segmento de cliente** · *Necesidad:* ¿qué segmento compra más y sufre más retrasos?
**Gráfico sugerido (+):** Barras · X = `customer_segment` · Y = `ventas` (y otro con `pct_tardias`).

In [ ]:
display(spark.table("gold_segmento"))

**7.18 Top 10 productos por ventas** · *Necesidad:* ¿cuáles son los productos estrella y qué tan puntuales son sus entregas?
**Gráfico sugerido (+):** Barras horizontales · X = `ventas` · Y = `product_name`.

In [ ]:
display(spark.table("gold_top_productos"))

**7.19 Estado del pedido** · *Necesidad:* ¿cómo se distribuyen los pedidos por estado, incluido el posible fraude?
**Gráfico sugerido (+):** Barras · X = `order_status` · Y = `pedidos_items`.

In [ ]:
display(spark.table("gold_estado_pedido"))

**7.20 Tipo de pago** · *Necesidad:* ¿el tipo de pago se relaciona con los retrasos?
**Gráfico sugerido (+):** Barras · X = `payment_type` · Y = `pct_tardias`.

In [ ]:
display(spark.table("gold_tipo_pago"))

**7.21 Distribución de la brecha de días (real − programado)** · *Necesidad:* ¿cuántos días se retrasan realmente las entregas? Positivo = retraso; negativo = adelanto.
**Gráfico sugerido (+):** Barras · X = `brecha_dias` · Y = `entregas`.

In [ ]:
display(spark.table("gold_brecha_dias"))

### F. Calidad de los datos

**7.22 Calidad de datos: Bronze vs. Silver** · *Necesidad:* ¿qué corrigió la limpieza?
**Gráfico sugerido (+):** Barras agrupadas · X = `metrica` · Y = `bronze` y `silver` (evidencia de la capa Silver).

In [ ]:
display(spark.table("gold_calidad"))

### Conclusiones del análisis
*Complete con lo que observe en sus tablas y gráficos:*
- Modo de envío con mayor porcentaje de retrasos: …
- Mercado y región con más entregas tardías: …
- Categorías o departamentos más afectados: …
- Tendencia mensual / anual: … (recuerde que 2018 solo tiene enero)
- Diferencia de ganancia entre entregas tardías y no tardías: …
- Recomendación para el negocio: …

> Con estas tablas y gráficos arme el **dashboard**.

## Parte 8. Reconciliación con la solución BI (CA-03) y regla de “tardía”
Se compara Spark con las cifras que el informe reporta para SQL Server / Power BI. **Tolerancia: 0.01.**

In [ ]:
tabla = spark.table(SILVER)

BI_Q1 = {"total_entregas": 180519, "entregas_tardias": 98977, "pct_tardias": 54.83,
         "dias_reales_prom": 3.50, "dias_prog_prom": 2.93}
BI_Q2 = {"First Class": (1.00, 2.00, 95.32), "Second Class": (2.00, 3.99, 76.63),
         "Same Day": (0.00, 0.48, 45.74), "Standard Class": (4.00, 4.00, 38.07)}   # (prog, real, % tardías)

fila = q1(tabla).collect()[0].asDict()
rec1 = pd.DataFrame([{"indicador": k, "spark": float(fila[k]), "bi": float(v),
                      "diferencia": round(float(fila[k]) - float(v), 4),
                      "cumple": abs(float(fila[k]) - float(v)) <= 0.01} for k, v in BI_Q1.items()])
display(rec1)

In [ ]:
q2_pd = q2(tabla).toPandas().set_index("shipping_mode")
filas = []
for modo, (prog, real, pct) in BI_Q2.items():
    s = q2_pd.loc[modo]
    for nombre, v_spark, v_bi in [("dias_prog", s.dias_prog_prom, prog), ("dias_reales", s.dias_reales_prom, real),
                                  ("pct_tardias", s.pct_tardias, pct)]:
        filas.append({"modo_envio": modo, "indicador": nombre, "spark": float(v_spark), "bi": v_bi,
                      "cumple": abs(float(v_spark) - v_bi) <= 0.01})
rec2 = pd.DataFrame(filas)
display(rec2)
print("CA-03 (Q1 y Q2 coinciden con el BI):", bool(rec1.cumple.all() and rec2.cumple.all()))

In [ ]:
# R4: bandera late_delivery_risk vs. regla días reales > programados
display(tabla.groupBy("late_delivery_risk", "entrega_tardia_regla").count().orderBy("late_delivery_risk", "entrega_tardia_regla"))
display(tabla.filter(F.col("late_delivery_risk") != F.col("entrega_tardia_regla"))
             .groupBy("delivery_status", "late_delivery_risk", "entrega_tardia_regla").count().orderBy(F.desc("count")))

**Para el informe (R4):** si las dos definiciones discrepan, reporte la cantidad exacta de filas y su `delivery_status`, tal como se prometió en la sección 4.7 del informe.

## Parte 9. EXP-02: escalabilidad del procesamiento
**Pregunta:** ¿cómo cambia el tiempo de Q1–Q4 y de la persistencia cuando el volumen crece de 1× a 16× el dataset original?

**Dataset de prueba / sintético (fila E-03):**
- *Método:* réplica de Silver `k` veces con `crossJoin`; `order_id` y `order_item_id` se desplazan `réplica × 1 000 000` para conservar la unicidad.
- *Escalas:* 1×, 2×, 4×, 8×, 16× (180 519 → ~2.9 M filas).
- *Límites:* las réplicas conservan la distribución original; los indicadores porcentuales no cambian. Solo se prueba el costo de procesamiento, no un crecimiento real del negocio.

**Medición:** 1 corrida de calentamiento descartada + **5 repeticiones**; se reportan mediana, media, desviación, **CV** y filas/s. Umbrales: T ≤ 5 s; CV ≤ 10 % (≤ 15 % si T < 1 s).

In [ ]:
meta = {"fecha": datetime.datetime.now().isoformat(timespec="seconds"), "spark": spark.version,
        "compute": "Databricks Free Edition (serverless)"}
print(meta)

ESCALAS = [1, 2, 4, 8, 16]
REPS = 5

def crear_escala(k):
    if k == 1:
        return df
    return (df.crossJoin(spark.range(k).withColumnRenamed("id", "replica"))
              .withColumn("order_id",      F.col("order_id")      + F.col("replica") * F.lit(1000000))
              .withColumn("order_item_id", F.col("order_item_id") + F.col("replica") * F.lit(1000000))
              .drop("replica"))

carga = []
for k in ESCALAS:
    nombre = f"tb_scale_{k}x"
    t0 = time.perf_counter()
    crear_escala(k).write.mode("overwrite").saveAsTable(nombre)
    seg = time.perf_counter() - t0
    n = spark.table(nombre).count()
    carga.append({"escala": f"{k}x", "tabla": nombre, "filas": n, "carga_s": round(seg, 2),
                  "throughput_carga_filas_s": round(n / seg)})
df_carga = pd.DataFrame(carga)
display(df_carga)

In [ ]:
def medir(fn, nombre_tabla, reps=REPS):
    t = spark.table(nombre_tabla)
    fn(t).collect()                       # calentamiento (descartado)
    tiempos = []
    for _ in range(reps):
        t0 = time.perf_counter()
        fn(t).collect()
        tiempos.append(time.perf_counter() - t0)
    return tiempos

resultados = []
for k in ESCALAS:
    nombre = f"tb_scale_{k}x"
    filas_k = int(df_carga.loc[df_carga.escala == f"{k}x", "filas"].iloc[0])
    for qn, fn in QUERIES.items():
        ts = medir(fn, nombre)
        med, media, sd = statistics.median(ts), statistics.mean(ts), statistics.stdev(ts)
        cv = sd / media * 100
        resultados.append({"escala": f"{k}x", "filas": filas_k, "consulta": qn,
                           "mediana_s": round(med, 3), "media_s": round(media, 3), "desv_s": round(sd, 3),
                           "cv_pct": round(cv, 1), "throughput_filas_s": round(filas_k / med),
                           "cumple_T_5s": med <= 5, "cumple_CV": cv <= (15 if med < 1 else 10),
                           "tiempos_s": [round(x, 3) for x in ts]})
df_res = pd.DataFrame(resultados)
display(df_res)

In [ ]:
resumen = (df_res.groupby(["escala", "filas"], as_index=False)
                 .agg(mediana_Q_s=("mediana_s", "median"), peor_Q_s=("mediana_s", "max"), cv_max_pct=("cv_pct", "max"))
                 .sort_values("filas").reset_index(drop=True))
resumen["veces_vs_1x"] = (resumen["mediana_Q_s"] / resumen.loc[0, "mediana_Q_s"]).round(2)
resumen["factor_volumen"] = (resumen["filas"] / resumen.loc[0, "filas"]).round(1)
display(resumen)

# Tabla para graficar con el +  →  Líneas · X = filas · Y = media_s · Series = consulta
display(df_res[["consulta", "escala", "filas", "media_s", "desv_s", "mediana_s", "cv_pct", "throughput_filas_s"]]
        .sort_values(["consulta", "filas"]))

In [ ]:
reg_df = df_res.drop(columns=["tiempos_s"]).copy()
reg_df["tiempos_s"] = df_res["tiempos_s"].astype(str)
reg_df["fecha"], reg_df["spark"] = meta["fecha"], meta["spark"]
spark.createDataFrame(reg_df).write.mode("overwrite").saveAsTable("tb_exp02_resultados")
print("Guardado en tb_exp02_resultados")

**Cómo interpretar (sin asumir el resultado):**
- Si el tiempo crece menos que el volumen (`veces_vs_1x` < `factor_volumen`), Spark absorbe parte del crecimiento.
- Con 180 519 filas el costo fijo de Spark puede dominar y SQL Server local podría ser más rápido en 1×. Repórtelo tal cual.
- Si algún CV supera el umbral, no lo oculte: explíquelo (cómputo serverless compartido).
- En un entorno gestionado no se controlan nodos ni memoria: documente la escalabilidad horizontal como no controlada por el equipo.

### EXP-03: comparativo BI vs. Big Data
No es una comparación equivalente (hardware y motor distintos); describe el comportamiento, no declara un ganador.

In [ ]:
BI_MEDIANA_S = 0.85           # EXP-1.1 del informe (SQL Server Express local)
comp = pd.DataFrame([
  {"solucion": "BI (SQL Server Express, local)",    "escala": "180 519 filas", "mediana_Q1_Q4_s": BI_MEDIANA_S},
  {"solucion": "Big Data (Spark, Databricks Free)", "escala": "1x",  "mediana_Q1_Q4_s": round(float(df_res[df_res.escala == "1x"]["mediana_s"].median()), 3)},
  {"solucion": "Big Data (Spark, Databricks Free)", "escala": "16x", "mediana_Q1_Q4_s": round(float(df_res[df_res.escala == "16x"]["mediana_s"].median()), 3)},
])
display(comp)

## Parte 10. (Opcional) Particionado entrenamiento/prueba y modelo
Igual que las Partes 6 y 10 de la guía, con `seed` para reproducibilidad.

In [ ]:
entrenamiento, prueba = df.randomSplit([0.8, 0.2], seed=42)
n_train, n_test = entrenamiento.count(), prueba.count()
print("Entrenamiento:", n_train, "| Prueba:", n_test, "| Suma:", n_train + n_test, "| Total:", df.count())

Modelo opcional (no está en los objetivos del informe). Predice `late_delivery_risk` con variables conocidas **al hacer el pedido**. No se usan `days_for_shipping_real`, `delivery_status` ni `entrega_tardia_regla` porque revelarían la respuesta (fuga de datos).

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

cat_cols = ["shipping_mode", "market", "customer_segment", "payment_type", "category_name"]
num_cols = ["days_for_shipment_scheduled", "order_item_quantity", "sales", "order_item_discount_rate"]

stages  = [StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep") for c in cat_cols]
stages += [OneHotEncoder(inputCols=[c + "_idx" for c in cat_cols], outputCols=[c + "_ohe" for c in cat_cols])]
stages += [VectorAssembler(inputCols=[c + "_ohe" for c in cat_cols] + num_cols, outputCol="features")]
stages += [RandomForestClassifier(labelCol="late_delivery_risk", featuresCol="features", numTrees=50, seed=42)]

modelo = Pipeline(stages=stages).fit(entrenamiento)
pred = modelo.transform(prueba)
auc = BinaryClassificationEvaluator(labelCol="late_delivery_risk", metricName="areaUnderROC").evaluate(pred)
print(f"AUC en prueba: {auc:.4f}")
display(pred.groupBy("late_delivery_risk", "prediction").count().orderBy("late_delivery_risk", "prediction"))

## Parte 11. Limpieza de tablas auxiliares y evidencias
Las tablas `tb_scale_*` ocupan almacenamiento. Ejecute la celda solo después de guardar la evidencia (capturas y CSV de `tb_exp02_resultados`).

| Evidencia | Contenido | Parte |
|---|---|---|
| Arquitectura | `Arquitectura_Medallion_DataCo.png` | 1 |
| Limpieza | Registro de pasos y reporte Bronze vs. Silver | 4 |
| Medallion | Tablas `bronze_*`, `silver_*`, `gold_*` en Catalog | 3, 4, 6 |
| Tablas y gráficos | Gráficos creados con el **+** y dashboard | 7 |
| E-08 | Resultados EXP-02 (`tb_exp02_resultados`) | 9 |
| E-14 | Reconciliación Spark vs. BI | 8 |

In [ ]:
# for k in ESCALAS:
#     spark.sql(f"DROP TABLE IF EXISTS tb_scale_{k}x")